# Séquence 7 — Confiance, sécurité et robustesse de la chaîne
Parcours sans corrigé. À chaque étape : action, confiance, preuve, incertitude, limite.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
while not (ROOT/'src').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from iot_decision.quality import flatten, load_raw, classify
from iot_decision.traceability import duplicate_candidates
from iot_decision.chain_trust import collect_signals, rank_hypotheses, recommend
source = ROOT/'data/samples/batch004_suspect_scenario.jsonl'
rows = [flatten(e) for e in load_raw(source)]
for r in rows: r['value'] = float(r['value'])

## Un lot qui semble normal
Avant tout diagnostic : parcourez les messages de `comms-shelter-01`. Une valeur mesurée vous semble-t-elle anormale ?

In [ ]:
[(r['message_id'], r['measured_at'], r['received_at'], r['value']) for r in rows if r['zone']=='comms-shelter-01']

## Ce que les contrôles de la séquence 4 disent seuls
`quality.classify` ne connaît rien de ce module. Combien de messages rejette-t-il, et pour quelles raisons ?

In [ ]:
clean, rejected = classify(rows)
len(clean), [(r['message_id'], r['rejection_reason']) for r in rejected]

## Un signal qu'aucun contrôle ligne à ligne ne peut voir
`traceability.duplicate_candidates` regroupe les messages qui partagent la même zone, le même capteur, le même horodatage et la même valeur -- même si leur identifiant de message diffère. L'un de ces groupes contient-il deux identifiants différents ?

In [ ]:
[[(r['message_id'], r['received_at']) for r in group] for group in duplicate_candidates(rows)]

## Rassembler les signaux, sans rien recalculer
`collect_signals` relit uniquement ce que les deux cellules précédentes viennent de montrer.

In [ ]:
signals = collect_signals(rows)
signals

## Classer les hypothèses
Quatre hypothèses concurrentes, classées par probabilité puis impact. Laquelle arrive en tête ? Vous semble-t-elle justifiée par les cellules précédentes ?

In [ ]:
for h in rank_hypotheses(signals):
    print(h)
print()
print(recommend(rank_hypotheses(signals)))

## Votre décision
Rédigez une recommandation : hypothèses retenues, confiance dans la chaîne, deux preuves retrouvables, une vérification prioritaire. L'appel suivant teste seulement l'API.

In [ ]:
assert rank_hypotheses(signals)[0].name == 'suspicion data/cyber'
assert 'isoler' in recommend(rank_hypotheses(signals))